# 07. Graph RAG — 논문에서 지식그래프를 뽑아 검색에 쓰기

`03`, `06` 에서 만든 **벡터 RAG** 는 "질문과 비슷한 문장이 있는 조각" 을 잘 찾습니다. 하지만 답이 **여러 조각·여러 논문에 흩어진 관계** 로 이루어져 있으면 약해집니다.
이번 노트북에서는 LLM 으로 논문 조각에서 **개체(재료·물성·현상…)와 관계** 를 뽑아 **지식그래프** 를 만들고(`04` 에서 배운 Kùzu 사용), 그 그래프를 검색에 활용하는 **Graph RAG** 를 직접 구현합니다.

**학습 목표**
1. 벡터 RAG 가 놓치는 질문 유형과 Graph RAG 의 전체 구조(인덱싱 → 검색 → 생성)를 설명할 수 있습니다.
2. 논문 조각을 **어휘 그래프(Paper–Chunk)** 로 저장하고, LLM 으로 **개체·관계를 JSON 으로 추출** 하는 프롬프트를 직접 작성합니다.
3. 추출 결과를 Kùzu 에 적재(MERGE)하고, 허브 개체·분포를 살펴봅니다.
4. 세 가지 그래프 검색기를 구현합니다: ① 개체 찾기 + 이웃 확장 ② **Text2Cypher** ③ 벡터 검색 → 그래프 확장(**VectorCypher**).
5. 같은 질문에 대해 벡터 RAG 와 Graph RAG 의 답을 비교하고 한계를 이해합니다.

> **실행 시간 안내**: 5절의 전체 개체·관계 추출이 가장 오래 걸립니다 (PC·모델에 따라 수십 분~수 시간). 나머지 절은 합쳐서 10~15분 정도입니다. 추출 결과는 `db/kg_triples.json` 에 저장되어 다음부터는 다시 할 필요가 없습니다.
>
> 이 노트북의 흐름은 공원나연 님의 「Neo4j GraphRAG 파이썬 패키지 가이드북」(neo4j-graphrag 패키지)을 참고했습니다. 여기서는 패키지 없이 **Kùzu + 순수 파이썬** 으로 같은 아이디어를 처음부터 만듭니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, re, shutil, textwrap, time
import kuzu, numpy as np, pandas as pd
from skku_rag import (DB_DIR, LLM_MODEL, EMBED_MODEL, get_llm, get_embeddings,
                      load_papers, split_documents)
# 그래프용 공용 도우미 (07 에서 만들고 08 에서도 다시 씁니다)
from skku_rag.graph_store import (ENTITY_TYPES, RELATION_TYPES, GRAPH_SCHEMA, TRIPLES_PATH,
                                  normalize_name, normalize_relation, paper_meta,
                                  make_chunk_records, vote_entity_types, run_cypher, is_read_only)

pd.set_option("display.max_colwidth", 80)
print("LLM:", LLM_MODEL, "| 임베딩:", EMBED_MODEL)

In [ ]:
llm = get_llm()   # 답변 생성·질문 분석에 쓸 LLM (temperature=0)

def show(text: str, width: int = 100):
    # 긴 답변을 보기 좋게 줄바꿈해서 출력
    print("\n".join(textwrap.fill(line, width) for line in str(text).splitlines()))

## 1. 벡터 RAG 가 놓치는 질문

다음 두 질문을 생각해 봅시다.

| 질문 | 왜 어려운가 |
|---|---|
| **"Cu 를 대체할 후보 금속들과 각각의 장단점은?"** | 예를 들어 Ru 의 장점, Co 의 단점, NiAl 의 특징이 서로 다른 쪽에 나오는 것처럼 답이 **여러 조각에 흩어져** 있음. 벡터 검색은 상위 k 개 조각만 보므로 일부만 답하게 됨 |
| **"결정립계 산란과 관련된 재료와 현상은?"** | "결정립계 산란 → 크기효과 → 비저항 증가 ← Cu" 처럼 **관계를 몇 단계 따라가야** 답이 모임 |

벡터 검색은 "**비슷한 문장**" 을 찾고, 그래프 검색은 "**연결된 사실**" 을 따라갑니다. 실험에 비유하면,
- 벡터 검색 = XRD 패턴이 **비슷한** 기준 시료를 데이터베이스에서 찾기
- 그래프 검색 = 상평형도에서 한 상과 **경계로 이웃한** 상들을 차례로 따라가기

## 2. Graph RAG 전체 그림

```
[인덱싱: 한 번만]                                         [검색·생성: 질문마다]
 논문 PDF                                                  질문 "Cu 대체 후보와 장단점?"
   │ 조각 나누기                                               │ ① 질문 속 개체 찾기 (LLM: "Cu", ALTERNATIVE_TO)
   ▼                                                          ▼
 Chunk ──PART_OF──▶ Paper      ← 어휘 그래프(lexical)        그래프에서 Cu 노드 찾기
   │ LLM 개체·관계 추출 (JSON)                                  │ ② 이웃 확장 (1~2 hop)
   ▼                                                          ▼
 Chunk ──MENTIONS──▶ Entity ──RELATED_TO──▶ Entity           관계 목록 + 관련 조각(원문)
                    (Ru)   {ALTERNATIVE_TO}  (Cu)               │ ③ LLM 에게 "이 근거로 답해"
   ▼ Kùzu 에 저장                                             ▼
 paper_graph.kuzu                                           답변 (+ 근거 조각 id)
```

`neo4j-graphrag` 패키지로 치면 인덱싱은 `SimpleKGPipeline`(조각 나누기 + `ERExtractionTemplate` 추출 + 저장), 검색은 `VectorRetriever / VectorCypherRetriever / HybridRetriever / Text2CypherRetriever`, 생성은 `GraphRAG + RagTemplate` 에 해당합니다. 아래에서 하나씩 직접 만들어 봅니다.

## 3. 어휘 그래프(Lexical Graph) 만들기: Paper – Chunk

먼저 LLM 없이 **"어떤 조각이 어느 논문의 몇 쪽인지"** 만 그래프로 저장합니다. 이것을 **어휘 그래프** 라고 부릅니다. 나중에 개체가 "어느 조각에서 나왔는지" 를 연결하는 뼈대가 됩니다.

개체·관계 추출은 조각 하나당 LLM 을 한 번 부르므로, 조각을 조금 크게(1500자) 나눠 호출 횟수를 줄입니다. (`CHUNK_SIZE` 는 공용 모듈에 정해 두었습니다. 08 노트북도 같은 값을 써야 조각 id 가 맞습니다.)

In [ ]:
from skku_rag.graph_store import CHUNK_SIZE, CHUNK_OVERLAP   # 1500자 / 150자 겹침

docs = load_papers()
chunks = split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
records = make_chunk_records(chunks)        # [{id, paper_id, page, text}, ...]
papers = {paper_meta(d.metadata["source"]) for d in docs}   # {(id, 제목, 연도)}

print("조각 수:", len(records))
print("논문:", sorted(p[0] for p in papers))
print("예시 조각:", {k: (v[:60] + "…" if k == "text" else v) for k, v in records[40].items()})

In [ ]:
db_path = DB_DIR / "paper_graph.kuzu"
# 셀을 다시 실행할 때: 열려 있던 연결·DB 를 먼저 닫아야 파일을 지울 수 있음 (Windows 파일 잠금)
if "conn" in globals():
    conn.close()
    db.close()
for p in [db_path, DB_DIR / "paper_graph.kuzu.wal"]:     # 깨끗하게 새로 시작
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
db = kuzu.Database(str(db_path))
conn = kuzu.Connection(db)

for stmt in [
    "CREATE NODE TABLE Paper(id STRING, title STRING, year INT64, PRIMARY KEY(id))",
    "CREATE NODE TABLE Chunk(id STRING, paper_id STRING, page INT64, text STRING, PRIMARY KEY(id))",
    "CREATE NODE TABLE Entity(name STRING, type STRING, PRIMARY KEY(name))",
    "CREATE REL TABLE PART_OF(FROM Chunk TO Paper)",
    "CREATE REL TABLE MENTIONS(FROM Chunk TO Entity)",
    "CREATE REL TABLE RELATED_TO(FROM Entity TO Entity, relation STRING, chunk_id STRING)",
]:
    conn.execute(stmt)

In [ ]:
for pid, title, year in papers:
    conn.execute("MERGE (p:Paper {id: $id}) SET p.title = $t, p.year = $y", {"id": pid, "t": title, "y": year})
for r in records:
    conn.execute("MERGE (c:Chunk {id: $id}) SET c.paper_id = $paper_id, c.page = $page, c.text = $text", r)
    conn.execute("MATCH (c:Chunk {id: $c}), (p:Paper {id: $p}) MERGE (c)-[:PART_OF]->(p)",
                 {"c": r["id"], "p": r["paper_id"]})

conn.execute('''
    MATCH (c:Chunk)-[:PART_OF]->(p:Paper)
    RETURN p.id AS 논문, p.year AS 연도, count(c) AS 조각수, max(c.page) AS 쪽수
''').get_as_df()

## 4. LLM 으로 개체·관계 추출하기

### 4-1. 추출 프롬프트 직접 작성

neo4j-graphrag 의 `ERExtractionTemplate` 처럼 **① 역할 ② 스키마(허용하는 개체·관계 종류) ③ 규칙 ④ 출력 JSON 예시 ⑤ 본문** 순서로 프롬프트를 만듭니다.
스키마를 주는 이유: 아무 제약이 없으면 LLM 이 조각마다 `"is_better_than"`, `"shows"`, `"exhibits"` 처럼 제각각인 관계 이름을 만들어 그래프가 연결되지 않습니다. **정해진 단어 목록** 을 주면 서로 다른 조각에서 나온 사실들이 같은 이름으로 모입니다.

공용 모듈에 정의된 개체·관계 종류는 다음과 같습니다.

In [ ]:
print("개체 종류:", ENTITY_TYPES)
print("관계 종류:", RELATION_TYPES)

프롬프트는 영어로 씁니다. 논문 원문이 영어이고, 소형 모델은 영어 지시를 더 잘 따르기 때문입니다. (`{{ }}` 는 파이썬 `format()` 에서 중괄호 글자 자체를 뜻합니다.)

In [ ]:
# 개체·관계 추출 프롬프트 (neo4j-graphrag 의 ERExtractionTemplate 을 참고해 직접 작성)
EXTRACTION_PROMPT = """You are a materials-science expert building a knowledge graph from a research paper.
Extract entities and relations from the TEXT below.

Entity types: {entity_types}
- Material: elements, alloys, compounds (e.g. Cu, Ru, NiAl, Cu-Nb, graphene)
- Property: measurable properties (e.g. resistivity, electron mean free path, melting point)
- Phenomenon: physical effects (e.g. size effect, grain boundary scattering, electromigration)
- Process: fabrication or treatment steps (e.g. sputtering, annealing, dry etching)
- Application: uses and devices (e.g. interconnect, barrier layer, transistor)
- Method: measurement or simulation techniques (e.g. TEM, DFT, four-point probe)

Relation types: {relation_types}

Rules:
- Use short English names. Chemical elements/compounds as formulas (Cu, Ru, CuAl2). Other names in lowercase singular.
- Only extract facts stated in the TEXT. Do not invent. At most 8 entities and 8 relations (the most important ones).
- Use ONLY the relation types listed above.
- source and target of every relation must also appear in entities.
- Return ONLY JSON in this format:
{{"entities": [{{"name": "Ru", "type": "Material"}}, {{"name": "Cu", "type": "Material"}}],
 "relations": [{{"source": "Ru", "relation": "ALTERNATIVE_TO", "target": "Cu"}}]}}

TEXT:
{text}
"""

### 4-2. 추출 함수와 JSON 파싱 실패 처리

LLM 은 가끔 JSON 앞뒤에 설명을 붙이거나(```` ```json ```` 등), 괄호를 빠뜨리는 등 **형식 오류** 를 냅니다. 그래서
1. 출력에서 `{ ... }` 부분만 정규식으로 잘라 내고,
2. 파싱에 실패하면 빈 결과로 처리한 뒤 **한 번 더 시도** 합니다.

> Ollama 의 `format="json"`(구조화 출력)을 쓰면 JSON 형식이 보장되지만, 실행 환경(예: Apple Silicon 의 MLX 엔진)에 따라 지원되지 않을 수 있어 여기서는 쓰지 않습니다.

In [ ]:
import json, re

extract_llm = get_llm()  # temperature=0: 같은 입력이면 최대한 같은 결과

def parse_json(raw: str) -> dict:
    """LLM 출력에서 JSON 부분만 꺼내 파싱합니다. 실패하면 빈 결과를 돌려줍니다."""
    match = re.search(r"\{.*\}", raw, re.S)  # 앞뒤 잡담·```json 표시를 무시하고 {...} 만
    try:
        data = json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        data = {}
    return {"entities": list(data.get("entities") or []), "relations": list(data.get("relations") or [])}

def extract_graph(text: str, retries: int = 1) -> dict:
    """조각 하나에서 개체·관계를 추출합니다. 결과가 비면 한 번 더 시도합니다."""
    prompt = EXTRACTION_PROMPT.format(
        entity_types=", ".join(ENTITY_TYPES), relation_types=", ".join(RELATION_TYPES), text=text)
    for _ in range(retries + 1):
        result = parse_json(extract_llm.invoke(prompt).content)
        if result["entities"]:
            break
    return result

In [ ]:
# 파싱 함수 시험: 잡담이 섞인 출력 / 망가진 출력
print(parse_json('다음과 같습니다:\n```json\n{"entities": [{"name": "Cu", "type": "Material"}], "relations": []}\n```'))
print(parse_json('{"entities": [{"name": "Cu", "type": "Mat'))   # 중간에 잘린 JSON → 빈 결과

### 4-3. 조각 2~3개에서 실제로 실행해 보기

Cu 합금(Kim 2016), 결정립계 산란 모델, Cu 대체 금속 이야기가 나오는 조각 3개를 골라 실행해 봅니다. (조각당 수십 초 걸릴 수 있습니다.)

In [ ]:
demo_ids = ["kim2016_c003", "sung2026_c013", "sung2026_c034"]   # Cu 합금 / 결정립계 산란 모델 / 대체 금속
demo_raw = {}
for cid in demo_ids:
    text = next(r["text"] for r in records if r["id"] == cid)
    t0 = time.time()
    demo_raw[cid] = extract_graph(text)
    print(f"[{cid}] {time.time() - t0:.0f}초 | 개체 {len(demo_raw[cid]['entities'])}개, 관계 {len(demo_raw[cid]['relations'])}개")

In [ ]:
cid = demo_ids[2]
print("원문 일부:", next(r["text"] for r in records if r["id"] == cid)[:500], "…\n")
print("추출된 관계:")
pd.DataFrame(demo_raw[cid]["relations"])

결과를 원문과 비교해 보세요. 4B 소형 모델은 다음 같은 실수를 자주 하니, 이런 것이 있는지 찾아보세요.
- 관계 방향을 거꾸로 쓰거나(`Cu -ALTERNATIVE_TO-> Ru`), 현상과 물성을 헷갈림
- 목록에 없는 관계 이름(`EXHIBITS`, `REQUIRES` …)을 지어냄
- 같은 대상을 `copper`, `Cu`, `Cu films` 처럼 다르게 부름

### 4-4. 결과 정리: 이름 정규화(Entity Resolution)

조각마다 같은 대상을 다르게 부르면 그래프에서 **다른 노드** 가 되어 연결이 끊깁니다. 시료 라벨을 통일하지 않으면 같은 시료의 XRD 와 SEM 데이터를 못 이어 붙이는 것과 같습니다. 공용 모듈의 `normalize_name()` 은 간단한 규칙으로 이름을 통일합니다.
- 별칭 사전: `copper → Cu`, `electron mean free path → EMFP`, `EM → electromigration` …
- 화학식(`Cu`, `NiAl`, `CuAl2`)은 대소문자 유지, 나머지는 소문자 + 단수형(`grain boundaries → grain boundary`)
- 목록에 없는 관계 이름은 `RELATED_TO` 로

In [ ]:
for raw in ["Copper", "Cu", "Grain Boundaries", "interconnects", "Electron mean free path", "NiAl", "ohmic losses"]:
    print(f"{raw!r:28} → {normalize_name(raw)!r}")
print("EXHIBITS →", normalize_relation("EXHIBITS"), "| alternative to →", normalize_relation("alternative to"))

In [ ]:
def clean_extraction(ex: dict) -> dict:
    """추출 결과 정리: 이름 정규화, 목록에 없는 type/관계 보정, 중복·자기 자신 관계 제거."""
    entities = {}
    for e in ex.get("entities", []):
        name = normalize_name(e.get("name", "")) if isinstance(e, dict) else ""
        if name:
            etype = e.get("type") if e.get("type") in ENTITY_TYPES else "Concept"
            entities.setdefault(name, etype)
    relations = []
    for r in ex.get("relations", []):
        if not isinstance(r, dict):
            continue
        src, dst = normalize_name(r.get("source", "")), normalize_name(r.get("target", ""))
        item = {"source": src, "relation": normalize_relation(r.get("relation", "")), "target": dst}
        if src and dst and src != dst and item not in relations:
            relations.append(item)
    return {"entities": [{"name": n, "type": t} for n, t in entities.items()], "relations": relations}

In [ ]:
pd.DataFrame(clean_extraction(demo_raw[demo_ids[2]])["relations"])

## 5. 전체 조각 추출 — 직접 실행 (시간이 걸립니다)

이제 같은 작업을 **참고문헌을 뺀 모든 본문 조각** 에 대해 실행합니다. (논문 끝의 참고문헌 목록 조각은 개체·관계가 거의 없어 건너뜁니다.) 4절에서 한 단계씩 만든 코드(프롬프트 → 파싱 → 재시도 → 정리)를 공용 모듈의 `extract_knowledge_graph()` 한 함수로 묶어 두었습니다. 하는 일은 똑같고, 다음 기능만 더했습니다.
- 조각마다 진행 상황 출력: `[12/101] sung2026_c010 개체 7개, 관계 5개 (35초)`
- 조각 하나 끝날 때마다 `db/kg_triples.json` 에 **바로 저장** → 중간에 멈추거나 Colab 연결이 끊겨도, 다시 실행하면 **끝난 조각은 건너뛰고 이어서** 합니다.

**예상 시간**: 조각 하나에 LLM 을 한 번 부르므로 전체 시간 = 조각 수(약 100개) × 조각당 시간입니다. 조각당 시간은 PC 사양과 모델에 따라 크게 다릅니다(GPU 가 없는 노트북은 느리고, Colab T4 GPU 는 보통 일반 노트북보다 빠릅니다). 처음에는 `MAX_CHUNKS = 30` 으로 실행해 진행 줄에 찍히는 조각당 시간을 보고, 전체 시간을 어림한 뒤 나머지를 이어서 실행하세요.

- `MAX_CHUNKS = None` : 남은 조각을 모두 추출합니다.
- **처음에는 `MAX_CHUNKS = 30` 으로 먼저 해 보세요(기본값).** 일부 조각만으로도 6절 이후를 모두 실행할 수 있습니다(그래프가 작아질 뿐입니다). 나중에 다시 `None` 으로 실행하면 나머지를 이어서 추출합니다.
- 모델이나 프롬프트를 바꿔 처음부터 다시 하려면 `db/kg_triples.json` 파일을 지우고 실행하세요.

In [ ]:
from skku_rag.graph_store import extract_knowledge_graph, reference_chunk_ids

skip = reference_chunk_ids(records)
print("전체 조각:", len(records), "| 참고문헌이라 건너뛸 조각:", len(skip), "| 추출 대상:", len(records) - len(skip))

In [ ]:
MAX_CHUNKS = 30      # 처음엔 30 으로 시험 → 조각당 시간을 확인한 뒤 None(전체)으로 바꿔 다시 실행 (이어서 추출)

t0 = time.time()
triples_path = extract_knowledge_graph(max_chunks=MAX_CHUNKS)   # 결과: db/kg_triples.json
print(f"이번 실행 {time.time() - t0:.0f}초")

In [ ]:
data = json.loads(triples_path.read_text(encoding="utf-8"))
meta = data["meta"]
print(f"추출 완료 조각: {meta['extracted_chunks']} / {meta['target_chunks']}  (모델 {meta['llm']})")
print(f"지금까지 추출에 쓴 시간 합계: {meta['elapsed_sec'] / 60:.0f}분")
assert {c["id"] for c in data["chunks"]} == {r["id"] for r in records}, "조각 나누기 설정이 추출 때와 다릅니다"

> 이미 끝까지 추출한 뒤 이 셀을 다시 실행하면 "이번에 0개 추출" 이 나오고 바로 끝납니다. 저장된 결과를 그대로 쓰는 것입니다.

## 6. Kùzu 에 적재하기 (MERGE)

1. **개체 노드**: 같은 개체가 조각마다 다른 type 으로 뽑혔을 수 있으므로(예: `resistivity` 를 Property 또는 Phenomenon) **가장 많이 나온 type** 으로 정합니다(다수결).
2. **MENTIONS**: 조각 → 그 조각에서 뽑힌 개체
3. **RELATED_TO**: 개체 → 개체. 관계 종류는 `relation` 속성에, **어느 조각에서 나온 사실인지** 는 `chunk_id` 속성에 저장합니다(출처 추적).

`MERGE` 를 쓰므로 같은 셀을 여러 번 실행해도 중복이 생기지 않습니다.

In [ ]:
entity_types = vote_entity_types(data["extractions"])     # {이름: 다수결 type}
for name, etype in entity_types.items():
    conn.execute("MERGE (e:Entity {name: $n}) SET e.type = $t", {"n": name, "t": etype})
print("개체 노드:", len(entity_types))

In [ ]:
for ex in data["extractions"]:
    cid = ex["chunk_id"]
    for rel in ex["relations"]:
        conn.execute('''MATCH (a:Entity {name: $s}), (b:Entity {name: $t})
                        MERGE (a)-[:RELATED_TO {relation: $r, chunk_id: $c}]->(b)''',
                     {"s": rel["source"], "t": rel["target"], "r": rel["relation"], "c": cid})
    mentioned = {e["name"] for e in ex["entities"]} | {x for rel in ex["relations"] for x in (rel["source"], rel["target"])}
    for name in mentioned:
        conn.execute("MATCH (c:Chunk {id: $c}), (e:Entity {name: $n}) MERGE (c)-[:MENTIONS]->(e)", {"c": cid, "n": name})

conn.execute("MATCH ()-[r:RELATED_TO]->() RETURN count(r) AS 관계수").get_as_df()

> 지금까지 한 일(스키마 생성 → 어휘 그래프 → 개체·관계 적재)은 공용 모듈의 `get_graph()` 한 줄에 그대로 들어 있습니다. `08_Agentic_RAG` 에서는 `from skku_rag.graph_store import get_graph; conn = get_graph()` 로 이 그래프를 바로 불러 씁니다.

## 7. 그래프 살펴보기

In [ ]:
# 개체 종류별 개수
conn.execute("MATCH (e:Entity) RETURN e.type AS 종류, count(*) AS 개수 ORDER BY 개수 DESC").get_as_df()

In [ ]:
# 관계 종류별 개수 — RELATED_TO(기타)가 많다면 LLM 이 목록 밖의 관계를 많이 만들었다는 뜻
conn.execute("MATCH ()-[r:RELATED_TO]->() RETURN r.relation AS 관계, count(*) AS 개수 ORDER BY 개수 DESC").get_as_df()

In [ ]:
# 허브(hub) 개체: 관계가 가장 많이 연결된 개체 = 두 논문의 핵심 주제
hubs = conn.execute('''
    MATCH (e:Entity)-[r:RELATED_TO]-(:Entity)
    RETURN e.name AS 개체, e.type AS 종류, count(r) AS 연결수
    ORDER BY 연결수 DESC LIMIT 12
''').get_as_df()
hubs

In [ ]:
# Cu 대체 후보라고 추출된 관계들 (어느 조각에서 나왔는지도 함께)
conn.execute('''
    MATCH (a:Entity)-[r:RELATED_TO]->(b:Entity {name: 'Cu'})
    WHERE r.relation = 'ALTERNATIVE_TO'
    RETURN a.name AS 후보, a.type AS 종류, collect(r.chunk_id) AS 출처조각
''').get_as_df()

### 7-1. 서브그래프 시각화

그래프 전체는 너무 커서, **Cu 주변(1 hop)** 의 관계 일부만 그려 봅니다.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

sub = conn.execute('''
    MATCH (a:Entity)-[r:RELATED_TO]->(b:Entity)
    WHERE a.name = 'Cu' OR b.name = 'Cu'
    RETURN DISTINCT a.name AS src, a.type AS src_type, r.relation AS rel, b.name AS dst, b.type AS dst_type
    LIMIT 30
''').get_as_df()
print("그릴 관계 수:", len(sub))

In [ ]:
colors = {"Material": "#4C9BE8", "Property": "#F2A541", "Phenomenon": "#E5625E", "Process": "#8CC084",
          "Application": "#B07CC6", "Method": "#7F8C8D", "Concept": "#CCCCCC"}
G = nx.DiGraph()
for row in sub.itertuples():
    G.add_node(row.src, type=row.src_type); G.add_node(row.dst, type=row.dst_type)
    G.add_edge(row.src, row.dst, rel=row.rel)

pos = nx.spring_layout(G, seed=1, k=1.2)
plt.figure(figsize=(13, 9))
nx.draw_networkx(G, pos, node_color=[colors.get(G.nodes[n]["type"], "#ccc") for n in G], node_size=700,
                 font_size=8, edge_color="#aaa", arrowsize=10)
nx.draw_networkx_edge_labels(G, pos, nx.get_edge_attributes(G, "rel"), font_size=6)
for t, c in colors.items():
    plt.scatter([], [], color=c, label=t)
plt.legend(loc="lower left", fontsize=8); plt.axis("off"); plt.title("Neighborhood of Cu (LLM-extracted)"); plt.show()

## 8. 검색 방법 ① — 질문 속 개체 찾기 + 이웃 확장

```
질문(한국어) ─LLM─▶ 영어 개체명 + 관계 힌트 ─▶ 그래프에서 이름 매칭 ─▶ 1~2 hop 이웃 관계 ─▶ 관계를 많이 담은 조각(원문)
"Cu 대체 후보?"      ["Cu"], ["ALTERNATIVE_TO"]     Cu 노드                 Ru→Cu, Co→Cu …           sung2026_c0xx …
```

**8-1. 질문에서 개체 뽑기**: 학생 질문은 한국어, 그래프 개체명은 영어입니다. LLM 에게 "질문 속 재료·개념을 그래프에 있을 법한 영어 이름으로" 바꾸게 합니다.

In [ ]:
QUERY_ENTITY_PROMPT = '''Find the key materials-science entities in the question (the question may be in Korean).
Write them as short English names used in papers: chemical formulas for elements/compounds (Cu, Ru, NiAl),
lowercase singular words for concepts (resistivity, grain boundary scattering, electromigration).
Also pick the relation types the question asks about from: {relation_types}
Return ONLY JSON: {{"entities": ["Cu"], "relations": ["ALTERNATIVE_TO"]}}

Question: {question}'''

def question_to_entities(question: str) -> dict:
    raw = llm.invoke(QUERY_ENTITY_PROMPT.format(relation_types=", ".join(RELATION_TYPES), question=question)).content
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        data = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        data = {}
    return {"entities": [str(e) for e in data.get("entities", [])], "relations": [str(r) for r in data.get("relations", [])]}

In [ ]:
Q1 = "Cu를 대체할 후보 금속들과 각각의 장단점은?"
Q2 = "결정립계 산란과 관련된 재료와 현상은?"
q1_info = question_to_entities(Q1)
q2_info = question_to_entities(Q2)
print("Q1 →", q1_info)
print("Q2 →", q2_info)

**8-2. 그래프에서 이름 매칭**: 정확히 같은 이름을 먼저, 그다음 이름에 키워드가 포함된 개체(예: `grain boundary` → `grain boundary scattering`)를 찾습니다. `Co`, `W` 처럼 짧은 이름은 `cost`, `low` 같은 엉뚱한 단어에 걸리지 않도록 정확일치만 허용합니다.

In [ ]:
def find_entities(keywords: list[str], limit: int = 8) -> pd.DataFrame:
    keys = [k for k in (normalize_name(k) for k in keywords) if k]
    if not keys:                                               # 찾을 이름이 없으면 빈 표 (빈 목록은 Kùzu 오류)
        return pd.DataFrame(columns=["name", "type", "degree", "exact"])
    long_keys = [k for k in keys if len(k) >= 4] or ["\x00"]   # 빈 목록이면 Kùzu 가 오류 → 절대 안 맞는 값
    return conn.execute('''
        MATCH (e:Entity)
        WHERE e.name IN $keys OR any(k IN $long_keys WHERE lower(e.name) CONTAINS lower(k))
        OPTIONAL MATCH (e)-[r:RELATED_TO]-(:Entity)
        RETURN e.name AS name, e.type AS type, count(r) AS degree, e.name IN $keys AS exact
        ORDER BY exact DESC, degree DESC LIMIT $limit
    ''', {"keys": keys, "long_keys": long_keys, "limit": limit}).get_as_df()

find_entities(q2_info["entities"])

**8-3. 이웃 확장**: 찾은 개체에서 한 단계씩 바깥으로 관계를 모읍니다(BFS). 2 hop 이상은 결과가 폭발적으로 늘어나므로, 각 단계에서 자주 등장한 개체 15개만 다음 단계로 넘깁니다. 질문이 원하는 관계(예: `ALTERNATIVE_TO`)는 앞쪽에 정렬합니다.

In [ ]:
def get_neighbors(names: list[str], hops: int = 1, relations: list[str] | None = None, limit: int = 40) -> pd.DataFrame:
    query = '''MATCH (a:Entity)-[r:RELATED_TO]->(b:Entity) WHERE a.name IN $f OR b.name IN $f
               RETURN a.name AS source, r.relation AS relation, b.name AS target, r.chunk_id AS chunk_id'''
    if not names:                                   # 찾은 개체가 없으면 빈 표
        return pd.DataFrame(columns=["source", "relation", "target", "chunk_id", "hop", "wanted"])
    frontier, seen, parts = list(names), set(names), []
    for hop in range(1, hops + 1):
        if not frontier:
            break
        edges = conn.execute(query, {"f": frontier}).get_as_df().assign(hop=hop)
        parts.append(edges)
        counts = pd.concat([edges.source, edges.target]).value_counts()
        frontier = [n for n in counts.index if n not in seen][:15]   # 새로 만난 개체 중 상위 15개
        seen |= set(frontier)
    result = pd.concat(parts).drop_duplicates(["source", "relation", "target"])
    result["wanted"] = result.relation.isin(relations or [])
    return result.sort_values(["wanted", "hop"], ascending=[False, True]).head(limit).reset_index(drop=True)

In [ ]:
q1_names = find_entities(q1_info["entities"]).name.tolist()
q1_edges = get_neighbors(q1_names, hops=1, relations=q1_info["relations"])
print("시작 개체:", q1_names, "| 모은 관계:", len(q1_edges))
q1_edges.head(12)

**8-4. 관련 조각(원문) 모으기**: 관계만으로는 "왜 장점인지" 같은 설명이 부족합니다. 모은 관계의 **출처 조각(chunk_id)** 중 관계가 많이 나온 조각을 골라 원문도 함께 LLM 에게 줍니다. 그래프가 "어디를 읽어야 할지" 알려 주는 **색인** 역할을 하는 셈입니다.

In [ ]:
def chunks_by_ids(chunk_ids: list[str]) -> pd.DataFrame:
    if not len(chunk_ids):                          # 빈 목록이면 Kùzu 가 오류 → 빈 표
        return pd.DataFrame(columns=["id", "paper", "page", "text"])
    return conn.execute('''MATCH (c:Chunk)-[:PART_OF]->(p:Paper) WHERE c.id IN $ids
                           RETURN c.id AS id, p.id AS paper, c.page AS page, c.text AS text''',
                        {"ids": list(chunk_ids)}).get_as_df()

def top_source_chunks(edges: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    top_ids = edges.chunk_id.value_counts().head(k).index.tolist()   # 관계를 가장 많이 제공한 조각 순
    return chunks_by_ids(top_ids).set_index("id").loc[top_ids].reset_index()

top_source_chunks(q1_edges)[["id", "paper", "page"]]

**8-5. 답변 생성 (RagTemplate)**

neo4j-graphrag 의 `RagTemplate` 은 `Answer the user question using the provided context. / Context / Examples / Question / Answer` 구조입니다. 같은 구조에 한국어 지시를 더해 씁니다. 컨텍스트에는 **① 그래프 관계 목록** 과 **② 원문 조각** 이 함께 들어갑니다.

In [ ]:
RAG_TEMPLATE = '''Answer the user question using the provided context.
아래 Context 만 근거로 한국어로 답하세요. Context 에 없는 내용은 지어내지 말고 "자료에 없음" 이라고 하세요.
문장 끝에 근거로 쓴 조각 id 를 대괄호로 표시하세요 (형식: [논문id_c번호]).

Context:
{context}

Examples:
{examples}

Question:
{question}

Answer:'''

def format_context(edges: pd.DataFrame, chunk_df: pd.DataFrame, max_chars: int = 1500) -> str:
    facts = "\n".join(f"- {r.source} -[{r.relation}]-> {r.target}  ({r.chunk_id})" for r in edges.itertuples())
    texts = "\n\n".join(f"[{r.id}] ({r.paper}, p.{r.page})\n{r.text[:max_chars]}" for r in chunk_df.itertuples())
    return f"## Knowledge graph facts\n{facts}\n\n## Source chunks\n{texts}"

In [ ]:
def graph_rag(question: str, hops: int = 1, k: int = 3, verbose: bool = True) -> str:
    info = question_to_entities(question)                       # ① 질문 → 개체·관계 힌트
    names = find_entities(info["entities"]).name.tolist()        # ② 그래프에서 이름 매칭
    edges = get_neighbors(names, hops=hops, relations=info["relations"], limit=30)   # ③ 이웃 확장
    chunk_df = top_source_chunks(edges, k=k)                    # ④ 근거 조각 (관계가 없으면 빈 표)
    if verbose:
        print(f"개체 {names} | 관계 {len(edges)}개 | 조각 {chunk_df.id.tolist()}")
    prompt = RAG_TEMPLATE.format(context=format_context(edges, chunk_df), examples="", question=question)
    return llm.invoke(prompt).content                            # ⑤ 답변 생성

In [ ]:
graph_answers = {}
graph_answers[Q1] = graph_rag(Q1)
show(graph_answers[Q1])

In [ ]:
graph_answers[Q2] = graph_rag(Q2, hops=2)    # 관계를 두 단계까지 따라가기
show(graph_answers[Q2])

## 9. 검색 방법 ② — Text2Cypher (LLM 이 Cypher 를 직접 작성)

`02` 의 Text2SQL 과 같은 아이디어입니다. LLM 에게 **그래프 스키마 + 예시 질문/쿼리(few-shot)** 를 주고, 질문을 Cypher 로 바꾸게 한 다음 실행합니다.
- 장점: "몇 개?", "어느 논문 몇 쪽?" 처럼 **정확한 구조적 질문** 에 강합니다.
- 위험: LLM 이 만든 쿼리가 데이터를 지우거나 바꿀 수 있으므로 **읽기 전용 검사** (`MATCH`·`RETURN` 등으로 시작하는 쿼리만 허용, `CREATE / DELETE / SET …` 금지)를 반드시 합니다. 문법 오류가 나면 **오류 메시지를 보여 주고 다시 쓰게** 합니다.

공용 모듈의 `GRAPH_SCHEMA` 는 LLM 에게 줄 스키마 설명입니다.

In [ ]:
print(GRAPH_SCHEMA)

In [ ]:
TEXT2CYPHER_PROMPT = '''Task: Write a Kuzu Cypher query that answers the question.
Schema:
{schema}
Rules: use only the tables and properties in the schema. Relation kinds are values of r.relation.
Read-only: use MATCH / WHERE / RETURN / ORDER BY / LIMIT only. Always add LIMIT 20.
Return ONLY the Cypher query, no explanation.

Examples:
Question: Ru 는 어떤 물성을 가지나?
Cypher: MATCH (m:Entity {{name: 'Ru'}})-[r:RELATED_TO]->(p:Entity) WHERE r.relation = 'HAS_PROPERTY' RETURN p.name LIMIT 20
Question: 크기효과(size effect)를 언급하는 조각은 몇 개인가?
Cypher: MATCH (c:Chunk)-[:MENTIONS]->(e:Entity {{name: 'size effect'}}) RETURN count(c) AS n LIMIT 20
Question: Co 가 나오는 논문과 쪽 번호는?
Cypher: MATCH (p:Paper)<-[:PART_OF]-(c:Chunk)-[:MENTIONS]->(e:Entity {{name: 'Co'}}) RETURN DISTINCT p.id, c.page ORDER BY c.page LIMIT 20
{error}
Question: {question}
Cypher:'''

In [ ]:
def clean_cypher(text: str) -> str:
    text = re.sub(r"```(?:cypher)?", "", text).strip()          # 코드블록 표시 제거
    return text.split("Cypher:")[-1].strip().rstrip(";")

def text2cypher(question: str, max_tries: int = 3):
    error = ""
    for attempt in range(1, max_tries + 1):
        prompt = TEXT2CYPHER_PROMPT.format(schema=GRAPH_SCHEMA, question=question, error=error)
        query = clean_cypher(llm.invoke(prompt).content)
        print(f"[시도 {attempt}] {query}")
        try:
            return query, run_cypher(conn, query)                 # run_cypher: 읽기 전용 검사 후 실행
        except Exception as e:                                     # 문법 오류·쓰기 쿼리 → 이유를 알려 주고 재시도
            error = f"\nYour previous query `{query}` failed: {str(e)[:200]}. Fix it.\n"
            print("   → 오류:", str(e)[:120])
    return query, pd.DataFrame()

In [ ]:
# 읽기 전용 검사 확인
print(is_read_only("MATCH (e:Entity) RETURN e.name"), is_read_only("MATCH (e:Entity) DETACH DELETE e"))

In [ ]:
query, result = text2cypher("비저항(resistivity)을 증가시키는(INCREASES) 것은 무엇인가?")
result

In [ ]:
query, result = text2cypher("NiAl 을 언급하는 조각은 어느 논문의 몇 쪽에 있나?")
result

쿼리 결과(표)를 다시 LLM 에게 주면 자연어 답변이 됩니다.

In [ ]:
answer = llm.invoke(RAG_TEMPLATE.format(
    context=f"Cypher: {query}\nResult:\n{result.to_string(index=False)}", examples="",
    question="NiAl 은 어느 논문 몇 쪽에서 다뤄지나요?")).content
show(answer)

> 소형 모델의 Text2Cypher 는 자주 틀립니다(없는 속성 이름, 관계 방향 오류, Neo4j 전용 함수 사용 등). 몇 번 다시 실행해 보면 결과가 달라지기도 합니다. **few-shot 예시를 질문 유형에 맞게 늘리는 것** 이 가장 효과적인 개선 방법이고, 실무에서는 더 큰 모델을 쓰거나 자주 쓰는 질의를 미리 함수로 만들어 둡니다(→ `08` 의 도구).

## 10. 검색 방법 ③ — 벡터 검색으로 시작해 그래프로 확장 (VectorCypher)

neo4j-graphrag 의 `VectorCypherRetriever` 아이디어입니다.
1. 질문과 **의미가 비슷한 조각** 을 벡터 검색으로 찾고 (`03` 에서 numpy 로 했던 방식),
2. 그 조각이 언급하는 개체에서 **그래프 관계를 따라** 다른 조각에서 나온 사실까지 덧붙입니다.

개체 이름을 몰라도 시작할 수 있고, 벡터 검색이 못 본 조각의 사실을 그래프가 보충해 줍니다.

In [ ]:
emb = get_embeddings()
t0 = time.time()
chunk_ids = [r["id"] for r in records]
chunk_vecs = np.array(emb.embed_documents([r["text"] for r in records]))   # 조각 전체 임베딩
chunk_vecs /= np.linalg.norm(chunk_vecs, axis=1, keepdims=True)            # 길이 1로 정규화 → 내적 = 코사인 유사도
print(f"임베딩 {chunk_vecs.shape} ({time.time() - t0:.0f}초)")

def vector_search(question: str, k: int = 3) -> list[str]:
    q = np.array(emb.embed_query(question)); q /= np.linalg.norm(q)
    return [chunk_ids[i] for i in np.argsort(chunk_vecs @ q)[::-1][:k]]

In [ ]:
def vector_cypher(question: str, k: int = 3, limit: int = 30):
    ids = vector_search(question, k)                     # ① 벡터 검색으로 시작 조각
    if not ids:                                          # 빈 목록이면 Kùzu 가 오류 → 빈 표
        return ids, pd.DataFrame(columns=["source", "relation", "target", "chunk_id", "from_other_chunk"])
    edges = conn.execute('''
        MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)-[r:RELATED_TO]->(n:Entity)
        WHERE c.id IN $ids
        RETURN DISTINCT e.name AS source, r.relation AS relation, n.name AS target, r.chunk_id AS chunk_id,
               (NOT r.chunk_id IN $ids) AS from_other_chunk
        ORDER BY from_other_chunk DESC LIMIT $limit
    ''', {"ids": ids, "limit": limit}).get_as_df()          # ② 그 조각의 개체 → 그래프 관계로 확장
    return ids, edges

ids, vc_edges = vector_cypher(Q2)
print("벡터 검색 조각:", ids, "| 그래프로 얻은 관계:", len(vc_edges), "| 그중 다른 조각에서 온 사실:", vc_edges.from_other_chunk.sum())
vc_edges.head(10)

In [ ]:
vc_answer = llm.invoke(RAG_TEMPLATE.format(
    context=format_context(vc_edges, chunks_by_ids(ids)), examples="", question=Q2)).content
show(vc_answer)

## 11. 벡터 RAG vs Graph RAG 비교

같은 두 질문을 **벡터 RAG**(질문과 비슷한 조각 3개만 근거, `03`/`06` 방식)로도 답하게 하고 나란히 놓아 봅니다. 답변 프롬프트는 똑같이 `RAG_TEMPLATE` 을 씁니다.

In [ ]:
def vector_rag(question: str, k: int = 3) -> str:
    chunk_df = chunks_by_ids(vector_search(question, k))
    texts = "\n\n".join(f"[{r.id}] ({r.paper}, p.{r.page})\n{r.text[:1500]}" for r in chunk_df.itertuples())
    return llm.invoke(RAG_TEMPLATE.format(context=texts, examples="", question=question)).content

vector_answers = {q: vector_rag(q) for q in [Q1, Q2]}

In [ ]:
compare = pd.DataFrame({
    "질문": [Q1, Q2],
    "벡터 RAG": [vector_answers[Q1], vector_answers[Q2]],
    "Graph RAG (방법 ①)": [graph_answers[Q1], graph_answers[Q2]],
})
compare.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"})

**읽는 법**
- 벡터 RAG 가 상위 3개 조각에 나온 후보만 말하는지 확인해 보세요. Graph RAG 는 그래프에 모인 `ALTERNATIVE_TO` 관계 덕분에 **여러 조각에 흩어진 후보** 를 한꺼번에 나열하기 쉽습니다.
- 반면 Graph RAG 의 답은 **추출 품질에 묶여** 있습니다. LLM 이 관계를 잘못 뽑았다면(방향이 반대, 엉뚱한 개체) 답도 틀립니다. 근거 조각 id 를 보고 원문을 꼭 확인하세요.
- 어느 쪽이 항상 낫지는 않습니다. "~란 무엇인가?" 같은 **설명형 질문은 벡터 RAG**, "~와 관련된 것들은?", "~의 후보를 모두" 같은 **관계·나열형 질문은 Graph RAG** 가 유리합니다. `08` 에서는 에이전트가 질문을 보고 둘 중 하나(또는 SQL)를 고르게 만듭니다.

## 12. 한계와 개선 방향

| 한계 | 원인 | 개선 방향 |
|---|---|---|
| 추출 오류 (방향 반대, 없는 사실) | 4B 소형 모델, 조각 하나만 보고 판단 | 더 큰 모델, 추출 후 검증 단계, 사람이 일부 검수 |
| 같은 대상이 다른 노드로 | 이름 표기 차이 (`Cu film`, `Cu interconnect`) | 별칭 사전 확장, 임베딩 유사도로 개체 병합(entity resolution) |
| 추출 비용·시간 | 조각마다 LLM 호출 (PC·모델에 따라 수십 분~수 시간) | 결과 저장·재사용, 더 빠른 GPU, 중요한 절만 추출 |
| 허브 개체 쏠림 | `Cu`, `resistivity` 에 관계가 몰림 | 관계 종류 필터, hop 제한, 관계 가중치 |
| Text2Cypher 오류 | 소형 모델의 쿼리 작성 능력 한계 | few-shot 예시 보강, 자주 쓰는 질의는 함수(도구)로 고정 |

## 정리

- **Graph RAG = 인덱싱(조각 → LLM 개체·관계 추출 → 그래프 저장) + 그래프 검색 + LLM 답변** 입니다.
- 어휘 그래프(Paper–Chunk)와 개체 그래프(Entity–RELATED_TO)를 `MENTIONS` 로 이어 두면, 관계에서 원문 조각으로 언제든 되돌아갈 수 있습니다(출처 추적).
- 검색 방법 세 가지

| 방법 | 시작점 | 잘하는 질문 | neo4j-graphrag 대응 |
|---|---|---|---|
| ① 개체 찾기 + 이웃 확장 | 질문 속 개체 이름 | 관계·나열형 ("~의 후보들", "~와 관련된 것") | (VectorCypher 의 그래프 확장 부분) |
| ② Text2Cypher | LLM 이 쓴 Cypher | 정확한 구조 질문 ("몇 개", "어느 논문 몇 쪽") | `Text2CypherRetriever` |
| ③ 벡터 → 그래프 확장 | 의미가 비슷한 조각 | 개체 이름을 모를 때 + 흩어진 사실 보충 | `VectorCypherRetriever` |

- 공용 모듈 `skku_rag/graph_store.py` 에 `get_graph()`, `find_entities()`, `get_neighbors()`, `chunks_for_entities()`, `run_cypher()`, `GRAPH_SCHEMA` 를 정리해 두었습니다. `08_Agentic_RAG` 에서 에이전트의 도구로 씁니다.

## 직접 해보기

1. `graph_rag()` 로 "Ru 의 장점과 단점은?" 을 물어보세요. `hops=1` 과 `hops=2` 의 답을 비교하고, 모은 관계 수와 답의 품질이 어떻게 달라지는지 살펴보세요.
2. `TEXT2CYPHER_PROMPT` 의 예시(few-shot)에 "Cu 를 대체할 수 있는 재료는?" 질문과 그 Cypher 를 추가한 뒤, "Cu 대체 후보 중 Material 종류인 것만 보여줘" 를 Text2Cypher 로 실행해 보세요.
3. `QUERY_ENTITY_PROMPT` 없이 질문을 그대로 `find_entities(["결정립계 산란"])` 에 넣으면 왜 아무것도 찾지 못할까요? 한국어 질문 → 영어 그래프의 연결 고리가 어디인지 설명해 보세요.

## 참고자료
- 공원나연(박나연), 「Neo4j GraphRAG 파이썬 패키지 가이드북」(WikiDocs): https://wikidocs.net/book/16760 — 어휘 그래프, ERExtractionTemplate 식 추출, Vector/VectorCypher/Hybrid/Text2Cypher 검색기, RagTemplate 구성을 참고했습니다.
- neo4j-graphrag-python: https://github.com/neo4j/neo4j-graphrag-python
- Microsoft GraphRAG (개념 참고): https://microsoft.github.io/graphrag/
- Kùzu 문서: https://docs.kuzudb.com/
- 논문 데이터: Kim et al., *Scripta Materialia* 123 (2016) 113–117 / Sung et al., *J. Mater. Chem. C* 14 (2026) 6558–6580

## 마무리: 그래프 DB 연결 닫기

08을 실행하기 전에 그래프 DB 연결을 닫습니다 (Kùzu는 한 번에 한 프로그램만 파일을 열 수 있음).

In [ ]:
conn.close()   # 연결 닫기
db.close()     # DB 파일 잠금 해제